In [1]:
import torch
import torch.nn as nn
import cv2
import numpy as np
from torchvision import transforms
from PIL import Image


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Linear(128, 1)
        self.bbox_head = nn.Linear(128, 4)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)

        obj_logit = self.classifier(x)
        bbox = torch.sigmoid(self.bbox_head(x))

        return obj_logit, bbox


In [4]:
model = PneumoniaCNN().to(device)
model.load_state_dict(torch.load(r"D:\D drive\Project\AI That Explains Medical Images Like a Doctor\Another way\model creation\pneumonia_cnn.pth", map_location=device))
model.eval()


C:\Users\deven\AppData\Local\Temp\ipykernel_21896\2795692269.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"D:\D drive\Project\AI Tha

PneumoniaCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Linear(in_features=128, out_features=1, bias=True)
  (bbox_head): Linear(in_features=128, out_features=4, bias=True)
)

In [5]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

def load_xray(image_path):
    image = Image.open(image_path).convert("L")
    image = transform(image)
    return image.unsqueeze(0).to(device)


In [12]:
def predict(model, image_tensor):
    with torch.no_grad():
        obj_logit, bbox = model(image_tensor)
        prob = torch.sigmoid(obj_logit).item()

    label = "Pneumonia" if prob > 0.35 else "Normal"
    return label, prob, bbox


In [7]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_tensor):
        self.model.zero_grad()

        # 🔹 Forward pass
        obj_logit, _ = self.model(input_tensor)

        # 🔹 Backprop ONLY from classification logit
        obj_logit.backward(torch.ones_like(obj_logit))

        # 🔹 Compute Grad-CAM
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)

        cam = torch.relu(cam)
        cam = cam.squeeze().detach().cpu().numpy()

        cam = cv2.resize(cam, (224, 224))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam


In [8]:
def detect_affected_lung(heatmap):
    h, w = heatmap.shape
    left = heatmap[:, :w//2].mean()
    right = heatmap[:, w//2:].mean()

    if abs(left - right) < 0.05:
        return "Both lungs"
    return "Right lung" if left > right else "Left lung"


def detect_opacity(heatmap, threshold=0.4):
    return (heatmap > threshold).sum() / heatmap.size > 0.05


def detect_spread(heatmap, threshold=0.4):
    binary = np.uint8(heatmap > threshold)
    components, _ = cv2.connectedComponents(binary)
    return "Localized" if components <= 3 else "Diffuse"


In [9]:
def full_pipeline(image_path):
    image_tensor = load_xray(image_path)

    label, confidence = predict(image_tensor)

    gradcam = GradCAM(model, target_layer)
    heatmap = gradcam.generate(image_tensor)

    explanation = {
        "prediction": label,
        "confidence": round(confidence, 2),
        "affected_lung": detect_affected_lung(heatmap),
        "opacity_detected": detect_opacity(heatmap),
        "spread_type": detect_spread(heatmap),
        "visual_patterns": []
    }

    if explanation["opacity_detected"]:
        explanation["visual_patterns"].extend([
            "increased opacity",
            "loss of normal lung texture"
        ])

    if explanation["spread_type"] == "Localized":
        explanation["visual_patterns"].append("localized lung involvement")
    else:
        explanation["visual_patterns"].append("diffuse lung involvement")

    return explanation, heatmap


In [13]:
# explanation_json, gradcam_map = full_pipeline(r"")
# 
# print(explanation_json)
image_tensor = load_xray(r"D:\D drive\Project\AI That Explains Medical Images Like a Doctor\Dataset\chest_xray\test\PNEUMONIA\person1_virus_6.jpeg")

label, confidence, bbox = predict(model, image_tensor)

gradcam = GradCAM(model, model.features[6])
heatmap = gradcam.generate(image_tensor)

explanation = extract_explanation_features(
    heatmap,
    label,
    confidence
)



RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[1, 1, 224, 224] to have 3 channels, but got 1 channels instead